# ETL INEGI_ATUS — Avance 03

# Proyecto A&C — Avance 03
**Análisis de accidentes y hechos de tránsito en la Ciudad de México**  
**Equipo:** Manuel Soto Romero y Araceli Liliana Reyes Cabello · UACM · Inteligencia de Negocios · Grupo 313 · 2026-2

## Objetivo
Procesar los CSV originales de INEGI-ATUS sin modificar los archivos fuente. El proceso conserva las columnas originales, identifica duplicados exactos, marca fechas no interpretables cuando existe una columna de fecha reconocible y genera archivos separados para conservados, pendientes, excluidos y duplicados.

**Pregunta de negocio:** ¿En qué alcaldías y franjas horarias de la Ciudad de México se registra la mayor cantidad de personas lesionadas en hechos de tránsito, de acuerdo con los registros disponibles a partir de 2018?

**Alcance:** INEGI-ATUS se trata como fuente estadística independiente. No se suman sus resultados con SSC o C5. No se inventan equivalencias entre municipios y alcaldías ni se imputan valores faltantes.

**Entrada esperada:** coloca los CSV originales en `data/raw/INEGI_ATUS/`.

**Búsqueda de entrada:** además de `data/raw`, este notebook localiza archivos originales en `Datos/` y en carpetas identificadas del Avance 02.


## 1. Entorno, rutas, bitácora y funciones auxiliares

La exportación se realiza en CSV comprimido (`.csv.gz`) para reducir el espacio utilizado. La verificación cuenta registros de manera secuencial, sin volver a cargar el archivo completo en memoria.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np
import hashlib
import re
import csv
import gzip
import unicodedata

TZ = ZoneInfo("America/Mexico_City")
from pathlib import Path

def localizar_raiz_proyecto():
    actual = Path.cwd().resolve()
    for carpeta in (actual, *actual.parents):
        if carpeta.name.lower() == "avance03":
            return carpeta
    raise FileNotFoundError(
        "No se encontró la carpeta Avance03 entre el directorio actual y sus carpetas superiores. "
        "Abre Jupyter desde Notebooks3/Avance03."
    )

ROOT = localizar_raiz_proyecto()
PROYECTO = ROOT.parent.parent
RAW = PROYECTO / "resultados" / "resultados_INEGI_ATUS" / "etapa3"
ARCHIVO_ATUS = RAW / "INEGI_ATUS_etapa3_conservados.csv"
if not ARCHIVO_ATUS.is_file():
    raise FileNotFoundError(f"No se encontró el archivo INEGI-ATUS conservado de etapa 3: {ARCHIVO_ATUS}")
ARCHIVOS = [ARCHIVO_ATUS]

RUN_ID = datetime.now(TZ).strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "data" / "processed" / "INEGI_ATUS" / RUN_ID
LOG_DIR = ROOT / "bitacora" / "INEGI_ATUS" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
BITACORA = []


def loggear(archivo, paso, regla, antes, despues, decision, justificacion, detalle=""):
    BITACORA.append({
        "fecha_hora_local": datetime.now(TZ).isoformat(timespec="seconds"),
        "fuente": "INEGI_ATUS",
        "archivo_origen": str(archivo),
        "proceso": paso,
        "regla": regla,
        "registros_antes": int(antes),
        "registros_despues": int(despues),
        "decision": decision,
        "justificacion": justificacion,
        "detalle": detalle,
        "version_proceso": "Avance03-v3"
    })


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def norm(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", "_", texto).strip("_")


def buscar_columna(df, aliases):
    mapa = {norm(col): col for col in df.columns}
    for alias in aliases:
        if norm(alias) in mapa:
            return mapa[norm(alias)]
    return None


def leer_csv_seguro(path):
    return pd.read_csv(
        path,
        dtype="string",
        encoding="utf-8-sig",
        low_memory=False
    )


def contar_registros_csv_gz(path):
    """Cuenta registros CSV lógicos, incluso si un texto contiene saltos de línea."""
    with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as archivo:
        lector = csv.reader(archivo)
        next(lector, None)  # encabezado
        return sum(1 for _ in lector)


def escribir(df, path):
    """Guarda por bloques en gzip y verifica el conteo sin cargar el CSV completo."""
    path = Path(path)
    if path.suffix != ".gz":
        path = path.with_suffix(path.suffix + ".gz")
    path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with gzip.open(path, "wt", encoding="utf-8-sig", newline="") as archivo:
            df.to_csv(
                archivo,
                index=False,
                chunksize=10000,
                quoting=csv.QUOTE_MINIMAL
            )

        if not path.exists() or path.stat().st_size == 0:
            raise OSError(f"El archivo no se creó correctamente: {path}")

        filas = contar_registros_csv_gz(path)
        if filas != len(df):
            raise AssertionError(
                f"Conteo incorrecto en {path.name}: se esperaban {len(df)} y se leyeron {filas}."
            )
        return path

    except Exception:
        if path.exists():
            path.unlink()
        raise



print("Proyecto:", PROYECTO)
print("Archivo de entrada INEGI-ATUS:", ARCHIVO_ATUS)
print("Archivos encontrados:", len(ARCHIVOS))
print("Carpeta de salida:", OUT)

## 2. Extracción e inventario

Cada archivo se mantiene como una tabla independiente porque sus unidades de observación pueden ser distintas. Se registra número de filas, columnas y huella SHA-256.

In [ ]:
inventario = []
tablas = {}

for ruta in ARCHIVOS:
    df = leer_csv_seguro(ruta)
    tablas[ruta.name] = df
    inventario.append({
        "archivo": ruta.name,
        "ruta": str(ruta),
        "filas_iniciales": len(df),
        "columnas": len(df.columns),
        "sha256": sha256(ruta),
        "columnas_detectadas": " | ".join(map(str, df.columns))
    })
    loggear(
        ruta.name, "Extracción", "RG-01/RG-11",
        len(df), len(df), "Conservar copia de trabajo",
        "El original no se modifica; se registra procedencia y huella SHA-256."
    )

inventario_df = pd.DataFrame(inventario)
display(inventario_df)
escribir(inventario_df, OUT / "inventario_fuentes.csv.gz")

## 3. Duplicados exactos y folios repetidos

Los duplicados exactos se identifican comparando todas las columnas originales. Se conserva la primera aparición y las copias adicionales se exportan aparte. Un folio repetido no se elimina automáticamente: se reporta para revisión y no se suma como categoría adicional en la conciliación.

In [ ]:
resultados = {}
RESUMEN_DUPLICADOS = []

ALIASES_FOLIO = [
    "folio", "numero_folio", "num_folio", "id_accidente",
    "id", "registro", "numero_de_accidente"
]

for nombre, original in tablas.items():
    entrada = len(original)
    mascara_duplicados = original.duplicated(keep="first")
    duplicados = original.loc[mascara_duplicados].copy()
    base = original.loc[~mascara_duplicados].copy()

    col_folio = buscar_columna(base, ALIASES_FOLIO)
    if col_folio:
        valores = base[col_folio].fillna("").astype("string").str.strip()
        mascara_folio = valores.ne("") & valores.duplicated(keep=False)
        folios_repetidos = base.loc[mascara_folio].copy()
        folios_repetidos["_motivo_revision"] = (
            "Folio repetido; se conserva el registro y se envía a revisión."
        )
        folios_repetidos["_regla_aplicada"] = "INEGI-R01"
    else:
        folios_repetidos = pd.DataFrame(
            columns=list(base.columns) + ["_motivo_revision", "_regla_aplicada"]
        )

    resultados[nombre] = {
        "base": base,
        "duplicados": duplicados,
        "folios_repetidos": folios_repetidos,
        "entrada": entrada,
        "columna_folio": col_folio
    }

    RESUMEN_DUPLICADOS.append({
        "archivo": nombre,
        "registros_iniciales": entrada,
        "duplicados_exactos_retirados": len(duplicados),
        "registros_unicos": len(base),
        "columna_folio_detectada": col_folio or "No identificada",
        "filas_con_folio_repetido": len(folios_repetidos)
    })

    loggear(
        nombre, "Duplicados exactos", "RG-07",
        entrada, len(base), "Retirar duplicados exactos",
        "Se conserva la primera aparición; folios repetidos se reportan aparte.",
        f"Duplicados exactos retirados: {len(duplicados)}"
    )

resumen_duplicados_df = pd.DataFrame(RESUMEN_DUPLICADOS)
display(resumen_duplicados_df)

## 4. Validación de fechas y clasificación

Cada registro único recibe una decisión primaria. Por defecto se conserva con una nota de alcance: solo se evalúan automáticamente las reglas detectables con las columnas disponibles. Si se identifica una fecha no vacía que no puede interpretarse, el registro pasa a `pendiente`. No se inventan reglas territoriales ni equivalencias geográficas.

In [ ]:
RESUMEN_VALIDACION = []
ALIASES_FECHA = [
    "fecha", "fecha_accidente", "fecha_ocurrencia", "fecha_evento",
    "fecha_del_accidente", "fecha_de_ocurrencia"
]

for nombre, resultado in resultados.items():
    base = resultado["base"].copy()
    base["_decision"] = "conservado"
    base["_motivo_decision"] = (
        "No se detectó incumplimiento en las reglas automáticas disponibles; "
        "revisar variables no mapeadas."
    )
    base["_regla_aplicada"] = "RG-10"
    base["_valores_ausentes"] = base.isna().sum(axis=1).astype("Int64")

    col_fecha = buscar_columna(base, ALIASES_FECHA)
    invalidas = 0

    if col_fecha:
        texto_fecha = base[col_fecha].fillna("").astype("string").str.strip()
        fecha_convertida = pd.to_datetime(
            texto_fecha.replace("", pd.NA),
            errors="coerce",
            dayfirst=True
        )
        mascara_fecha_invalida = texto_fecha.ne("") & fecha_convertida.isna()
        invalidas = int(mascara_fecha_invalida.sum())

        base.loc[mascara_fecha_invalida, "_decision"] = "pendiente"
        base.loc[mascara_fecha_invalida, "_motivo_decision"] = (
            "Fecha no interpretable; no se corrige por suposición."
        )
        base.loc[mascara_fecha_invalida, "_regla_aplicada"] = "INEGI-R02"

    resultado["base"] = base

    RESUMEN_VALIDACION.append({
        "archivo": nombre,
        "registros_unicos": len(base),
        "columna_fecha_detectada": col_fecha or "No identificada",
        "fechas_invalidas": invalidas,
        "conservados_provisionales": int(base["_decision"].eq("conservado").sum()),
        "pendientes": int(base["_decision"].eq("pendiente").sum()),
        "excluidos": int(base["_decision"].eq("excluido").sum())
    })

    loggear(
        nombre, "Validación de fechas", "INEGI-R02",
        len(base), len(base), "Clasificar registros",
        "Solo se marcan fechas no vacías e ininterpretables cuando se detecta una columna de fecha.",
        f"Columna: {col_fecha or 'No identificada'}; fechas inválidas: {invalidas}"
    )

resumen_validacion_df = pd.DataFrame(RESUMEN_VALIDACION)
display(resumen_validacion_df)

## 5. Exportación de resultados y conciliación

Las categorías son mutuamente excluyentes: conservados + pendientes + excluidos + duplicados exactos retirados = registros iniciales. Los folios repetidos son una marca superpuesta y no se suman al total. Los archivos se guardan comprimidos para reducir el uso de almacenamiento.

In [ ]:
controles = []

for nombre, resultado in resultados.items():
    base = resultado["base"]
    conservados = base.loc[base["_decision"].eq("conservado")].copy()
    pendientes = base.loc[base["_decision"].eq("pendiente")].copy()
    excluidos = base.loc[base["_decision"].eq("excluido")].copy()
    duplicados = resultado["duplicados"]
    folios_repetidos = resultado["folios_repetidos"]

    prefijo = Path(nombre).stem
    escribir(conservados, OUT / f"{prefijo}_limpio.csv.gz")
    escribir(pendientes, OUT / f"{prefijo}_pendientes.csv.gz")
    escribir(excluidos, OUT / f"{prefijo}_excluidos_por_regla.csv.gz")
    escribir(duplicados, OUT / f"{prefijo}_duplicados_exactos.csv.gz")
    escribir(folios_repetidos, OUT / f"{prefijo}_folios_repetidos_revision.csv.gz")

    total = len(conservados) + len(pendientes) + len(excluidos) + len(duplicados)
    correcto = total == resultado["entrada"]

    controles.append({
        "archivo": nombre,
        "registros_iniciales": resultado["entrada"],
        "duplicados_exactos_retirados": len(duplicados),
        "conservados": len(conservados),
        "pendientes": len(pendientes),
        "excluidos": len(excluidos),
        "total_reconciliado": total,
        "filas_con_folio_repetido_revision": len(folios_repetidos),
        "conciliacion_correcta": correcto
    })

    if not correcto:
        raise AssertionError(
            f"Conciliación incorrecta en {nombre}: {total} != {resultado['entrada']}"
        )

    loggear(
        nombre, "Conciliación", "RG-07/RG-11/RG-12",
        resultado["entrada"], total, "Validar destino de cada fila",
        "Conservados, pendientes, excluidos y duplicados exactos son categorías excluyentes."
    )

control_df = pd.DataFrame(controles)
display(control_df)
escribir(control_df, OUT / "conciliacion_por_archivo.csv.gz")
escribir(resumen_duplicados_df, OUT / "resumen_duplicados.csv.gz")
escribir(resumen_validacion_df, OUT / "resumen_validacion.csv.gz")

bitacora_df = pd.DataFrame(BITACORA)
escribir(bitacora_df, OUT / "bitacora.csv.gz")
escribir(bitacora_df, LOG_DIR / "bitacora.csv.gz")

print("\nETL INEGI-ATUS terminado.")
print("Resultados:", OUT)
print("Bitácora:", LOG_DIR / "bitacora.csv.gz")

## 6. Interpretación y limitaciones

Los conteos definitivos son los generados al ejecutar este notebook con los CSV originales de INEGI-ATUS. Las cifras de Avance 02 son antecedentes que deben volver a comprobarse, no resultados que se copian como si fueran actuales.

La clasificación `conservado` significa que no se activó una regla automática disponible en este notebook; no sustituye la revisión de variables que requieran catálogos o reglas específicas. La detección de folios repetidos es informativa y no elimina registros por sí sola.